# FOI – 230724012 – Number of Mothers/Fathers applied for a CAO application in Wales

## 1. Import packages and set options 
<a name="import_packages"></a>

In [ ]:
import pandas as pd  # a module which provides the data structures and functions to store and manipulate tables in dataframes
import pydbtools as pydb  # A module which allows SQL queries to be run on the Analytical Platform from Python, see https://github.com/moj-analytical-services/pydbtools
import boto3  # allows you to directly create, update, and delete AWS resources from Python scripts
import numpy as np
import re

#import the pacakges required to produce an excel spreadsheet
import xlsxwriter
import boto3, io, os, pathlib

# sets parameters to view dataframes for tables easier
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 900)
pd.set_option("display.max_colwidth", 200)

## 2. Define key variables to be used throughout the notebook 
<a name="define_key_variables"></a>

In [ ]:
#this is the database we will be extracting from
database = "familyman_dev_v3"

#this extracts the February snapshot from athena
snapshot_date = "2023-05-10"

#this is the athena database we will be storing our tables in
fcsq_database = "fcsq"

#this is the s3 bucket we will be saving data to
s3 = boto3.resource("s3")
bucket = s3.Bucket("alpha-family-data")

#setting current year and quarter
pub_quarter = 1
pub_year = 2023

## FCSQ Check

In [ ]:
# Making sure the number of C100 orders applied for match the CSV Children National 2023 Q1
    # Filter by type = Application, count_type = Order type, and order_type_code = (29 or 30 or 31 or 32)

# The fcsq.ca_apps_order_types removes the child ID and groups up so we only count an order type within an event once, rather than one per child

check1 = pydb.read_sql_query("SELECT year, count(*) as n  FROM fcsq.ca_apps_order_types WHERE year > 2010 and (order_code = 29 OR order_code = 30 OR order_code = 31 OR order_code = 32) group by year order by year")
check1

In [ ]:
# Checking the number of rows in the athena table 'ca_apps_order_types'
check2 = pydb.read_sql_query("SELECT * FROM fcsq.ca_apps_order_types")
check2

## Extract applicant information

In [ ]:
#Joined three familyman_dev_v3 tables together to extract case_number, role_model, role, field_model, and value columns to help produce the final outputs
#Filtered using the snapshot_date = "2023-05-10" across all three tables and filtered for b.ROLE_MODEL = 'APLC and a.FIELD_MODEL = 'APLC_RC' 
pydb.create_temp_table(
f""" 
SELECT c.case_number, 
    b.role_model, 
    a.role, 
    a.field_model, 
    a.value

FROM {database}.role_fields a
LEFT JOIN {database}.field_models b
ON a.field_model = b.field_model
LEFT JOIN {database}.roles c
ON a.role = c.role

WHERE a.FIELD_MODEL = 'APLC_RC'
AND b.ROLE_MODEL = 'APLC' 
AND a.mojap_snapshot_date = DATE('{snapshot_date}')
AND b.mojap_snapshot_date = DATE('{snapshot_date}')
AND c.mojap_snapshot_date = DATE('{snapshot_date}');
""",

"applicant_data")

In [ ]:
applicant_data = pydb.read_sql_query("SELECT * FROM __temp__.applicant_data")
applicant_data

## Extract respondent information

In [ ]:
#Joined three familyman_dev_v3 tables together to extract case_number, role_model, role, field_model, and value columns to help produce the final outputs
#Filtered using the snapshot_date = "2023-05-10" across all three tables and filtered for b.ROLE_MODEL = 'APLC and a.FIELD_MODEL = 'APLC_RC' 
pydb.create_temp_table(
f""" 
SELECT c.case_number, 
    b.role_model, 
    a.role, 
    a.field_model, 
    a.value

FROM {database}.role_fields a
LEFT JOIN {database}.field_models b
ON a.field_model = b.field_model
LEFT JOIN {database}.roles c
ON a.role = c.role

WHERE a.FIELD_MODEL = 'RSPC_RC'
AND b.ROLE_MODEL = 'RSPC' 
AND a.mojap_snapshot_date = DATE('{snapshot_date}')
AND b.mojap_snapshot_date = DATE('{snapshot_date}')
AND c.mojap_snapshot_date = DATE('{snapshot_date}');
""",

"respondent_data")

In [ ]:
respondent_data = pydb.read_sql_query("SELECT * FROM __temp__.respondent_data")
respondent_data

In [ ]:
pydb.create_temp_table( 
f"""
  SELECT a.year,
  a.quarter,
  a.receipt_date,
  a.case_number,
  a.event,
  a.order_case_type,
  a.order_type,
  a.order_code,
  a.order_desc,
  a.app_region,
  c.case_role as app,
  c.gender_desc as app_gender,
  r.case_role as resp,
  r.gender_desc as resp_gender,
  c.applicant_count
  
  FROM
       fcsq.ca_apps_order_types a 
    INNER JOIN fcsq.ca_applicants c
     on a.case_number = c.case_number AND a.receipt_date = c.min_of_receipt_date
    INNER JOIN fcsq.ca_respondents r 
     on a.case_number = r.case_number AND c.applicant_count = r.respondent_count
  
  WHERE a.year > 2012 
  AND a.year < 2023
  AND a.app_region = 'WALES'
  AND (a.order_code = 29 or a.order_code = 30 or a.order_code = 31 or a.order_code = 32)
  AND c.applicant_count = 1
""",

"final_table")

In [ ]:
check = pydb.read_sql_query("SELECT * FROM __temp__.final_table")
check

## Orders Applied For

In [ ]:
female_orders_applied_for = pydb.read_sql_query("SELECT year, count(*) as n FROM __temp__.final_table WHERE app_gender = 'Female' and resp_gender = 'Male' GROUP BY year ORDER BY year")
female_orders_applied_for

In [ ]:
male_orders_applied_for = pydb.read_sql_query("SELECT year, count(*) as n FROM __temp__.final_table WHERE app_gender = 'Male' and resp_gender = 'Female' GROUP BY year ORDER BY year")
male_orders_applied_for

## Applications

In [ ]:
female_app = pydb.read_sql_query("SELECT year, count(*) as n FROM (SELECT DISTINCT year, quarter, receipt_date, case_number, event, app_region, app, app_gender, resp, resp_gender FROM __temp__.final_table GROUP BY year, quarter, receipt_date, case_number, event, app_region, app, app_gender, resp, resp_gender) WHERE app_gender = 'Female' and resp_gender = 'Male' GROUP BY year ORDER BY year")
female_app

In [ ]:
male_app = pydb.read_sql_query("SELECT year, count(*) as n FROM (SELECT DISTINCT year, quarter, receipt_date, case_number, event, app_region, app, app_gender, resp, resp_gender FROM __temp__.final_table GROUP BY year, quarter, receipt_date, case_number, event, app_region, app, app_gender, resp, resp_gender) WHERE app_gender = 'Male' and resp_gender = 'Female' GROUP BY year ORDER BY year")
male_app

## Create Final Output

In [ ]:
#creating a panda dataframe for both order and applicant tables
#uses the queries underneath both order and applicant table
df1 = pd.DataFrame(female_orders_applied_for)

df2 = pd.DataFrame(male_orders_applied_for)

df3 = pd.DataFrame(female_app)

df4 = pd.DataFrame(male_app)

In [ ]:
#creates a seperate sheet for each dataframe 
with io.BytesIO() as xlsx_data:
    with pd.ExcelWriter(xlsx_data) as writer:
        
        df1.to_excel(writer, sheet_name='female_orders_applied_for', index=False)
        
        df2.to_excel(writer, sheet_name='male_orders_applied_for', index=False)
        
        df3.to_excel(writer, sheet_name='female_app', index=False) 
        
        df4.to_excel(writer, sheet_name='male_app', index=False) 
        

    ## Upload the in-memory data to S3
    s3 = boto3.resource("s3")
    bucket = s3.Bucket("alpha-family-data")
    object_name = "FOIs/FOI 230724012 – working table.xlsx"
    s3.Bucket("alpha-family-data").put_object(Key=object_name, Body=xlsx_data.getvalue())